# Rigorous Statistical Re-Analysis of HOWP FOLIO Experiment

This notebook performs a complete statistical re-analysis of the **Heterogeneous-Oracle World-Probing (HOWP)** experiment run on the FOLIO validation dataset.

**What this evaluates:**
- 7 prediction conditions (main method + baselines + ablations) on 204 FOLIO examples
- Per-condition accuracy with bootstrap 95% confidence intervals (10,000 resamples)
- McNemar's pairwise significance tests (15 pairs) with Benjamini-Hochberg FDR correction
- Normalized self-consistency (alpha-renaming bound variables before Jaccard clustering)
- Uncertain-class decomposition (coincidence hits vs genuine reasoning)
- Cross-world agreement AUC-ROC and Kendall τ
- Inter-model agreement rate (hetero vs homo oracle)

**Key finding:** All 7 conditions achieve ~20-25% accuracy with widely overlapping CIs — no condition is statistically distinguishable. The bottleneck is the ~38% beam recall ceiling from Llama-3.1-8B's FOL generation quality.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab
_pip('loguru==0.7.3')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import gc
import json
import math
import os
import re
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy.stats import chi2, binom, kendalltau
from sklearn.metrics import roc_auc_score
from loguru import logger

logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')

1

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ad782-beam-recall-as-the-binding-constraint-an/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f: return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

In [4]:
data = load_data()

## Configuration

Tunable parameters. `N_BOOTSTRAP` controls the number of bootstrap resamples for CI estimation — increase to 10,000 for the full analysis.

In [5]:
# Tunable parameters
N_BOOTSTRAP = 500  # original: 10_000 — reduce for faster demo

CONDITIONS = [
    'predict_main_method',
    'predict_top1_baseline',
    'predict_self_consistency',
    'predict_direct_judge',
    'predict_ablation_same_oracle',
    'predict_ablation_random_worlds',
    'predict_ablation_m4',
]
SHORT_NAMES = {
    'predict_main_method': 'Hetero-Oracle',
    'predict_top1_baseline': 'Top-1',
    'predict_self_consistency': 'Self-Consist',
    'predict_direct_judge': 'Direct-Judge',
    'predict_ablation_same_oracle': 'Same-Oracle',
    'predict_ablation_random_worlds': 'Rand-Worlds',
    'predict_ablation_m4': 'm=4 Worlds',
}

RNG = np.random.default_rng(42)

## Helper Functions

Core utilities: correctness check, bootstrap CI, Cohen's h effect size, Benjamini-Hochberg FDR correction, and FOL alpha-renaming for normalized self-consistency.

In [6]:
def is_correct(pred: str, gold: str) -> bool:
    return pred != '' and pred == gold


def bootstrap_ci(correct_arr: np.ndarray, n: int = N_BOOTSTRAP) -> tuple:
    """Return (lo, hi) 95% CI via bootstrap resampling."""
    means = np.empty(n)
    size = len(correct_arr)
    for i in range(n):
        idx = RNG.integers(0, size, size)
        means[i] = correct_arr[idx].mean()
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def cohen_h(p1: float, p2: float) -> float:
    return 2 * math.asin(math.sqrt(p1)) - 2 * math.asin(math.sqrt(p2))


def bh_correction(pvalues: list) -> list:
    """Benjamini-Hochberg FDR correction."""
    n = len(pvalues)
    order = np.argsort(pvalues)
    pvalues_arr = np.array(pvalues)
    adjusted = np.zeros(n)
    for rank, idx in enumerate(order, start=1):
        adjusted[idx] = pvalues_arr[idx] * n / rank
    # enforce monotonicity (running minimum from largest)
    min_so_far = 1.0
    for idx in reversed(order):
        if adjusted[idx] > min_so_far:
            adjusted[idx] = min_so_far
        else:
            min_so_far = adjusted[idx]
    return [min(1.0, float(v)) for v in adjusted]


def alpha_rename_fol(formula: str) -> str:
    """
    Alpha-rename all bound variables to x0, x1, x2... in order of first binding.
    Handles: forall X. P, exists X. P, lambda X. P, plus parenthesized binders.
    """
    # Find all binder occurrences: 'forall X', 'exists X', 'lambda X'
    binder_re = re.compile(r'\b(forall|exists|lambda)\s+([A-Za-z_][A-Za-z0-9_]*)\b')
    # Map from original var name -> renamed var
    mapping = {}
    counter = [0]

    def replace_binder(m):
        quant, var = m.group(1), m.group(2)
        if var not in mapping:
            mapping[var] = f'x{counter[0]}'
            counter[0] += 1
        return f'{quant} {mapping[var]}'

    renamed = binder_re.sub(replace_binder, formula)

    # Replace free occurrences of each original variable
    def replace_free(text: str, old: str, new: str) -> str:
        return re.sub(r'\b' + re.escape(old) + r'\b', new, text)

    for old, new in mapping.items():
        renamed = replace_free(renamed, old, new)

    return renamed.strip().lower()


def normalize_fol_candidate(candidate: dict) -> str:
    """Normalize a candidate dict to a single comparable string."""
    parts = []
    for p in candidate.get('premises_fol', []):
        parts.append(alpha_rename_fol(str(p)))
    parts.append(alpha_rename_fol(str(candidate.get('conclusion_fol', ''))))
    s = ' '.join(parts)
    return re.sub(r'\s+', '', s)


def jaccard_similarity(a: str, b: str) -> float:
    ta, tb = set(a), set(b)
    if not ta and not tb:
        return 1.0
    inter = len(ta & tb)
    union = len(ta | tb)
    return inter / union if union > 0 else 0.0


def cluster_and_pick(candidates_str: str) -> int:
    """
    Parse candidates JSON, cluster by Jaccard token overlap, return index of
    centroid of largest cluster. Returns -1 if parsing fails.
    """
    try:
        candidates = json.loads(candidates_str)
    except Exception:
        return -1
    n = len(candidates)
    if n == 0:
        return -1

    norms = [normalize_fol_candidate(c) for c in candidates]

    # Similarity matrix
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            s = jaccard_similarity(norms[i], norms[j])
            sim[i, j] = s
            sim[j, i] = s

    # Greedy single-link clustering: threshold = 0.3 (same as original approach)
    threshold = 0.3
    clusters = []
    assigned = [-1] * n
    for i in range(n):
        best_cluster = -1
        for ci, cl in enumerate(clusters):
            if any(sim[i, j] >= threshold for j in cl):
                best_cluster = ci
                break
        if best_cluster == -1:
            clusters.append([i])
            assigned[i] = len(clusters) - 1
        else:
            clusters[best_cluster].append(i)
            assigned[i] = best_cluster

    # Largest cluster (ties broken by lowest cluster index → lowest member index)
    largest_ci = max(range(len(clusters)), key=lambda ci: (len(clusters[ci]), -ci))
    cluster_members = clusters[largest_ci]

    # Centroid = member with highest average similarity to rest of cluster
    if len(cluster_members) == 1:
        return cluster_members[0]

    best_idx = min(cluster_members, key=lambda i: -np.mean([sim[i, j] for j in cluster_members]))
    return best_idx

## Load Examples

Extract the list of examples from the loaded dataset. Each example has predictions for all 7 conditions, the gold label, candidate FOL formulas, and world agreement scores.

In [7]:
examples = data['datasets'][0]['examples']
n_total = len(examples)
logger.info(f'Loaded {n_total} examples')

# Quick sanity check
golds_all = [ex['metadata_gold_label'] for ex in examples]
from collections import Counter
print('Gold label distribution:', Counter(golds_all))

09:54:34|INFO   |Loaded 99 examples


Gold label distribution: Counter({'Entailment': 33, 'Contradiction': 33, 'Uncertain': 33})


## Metric 1: Per-Condition Accuracy + Bootstrap 95% CIs

For each of the 7 conditions, compute accuracy and bootstrap 95% confidence intervals. Also computes per-label breakdown for the main method.

In [8]:
logger.info('Computing per-condition accuracy + bootstrap CIs')
per_condition = {}

for cond in CONDITIONS:
    preds = [ex.get(cond, '') for ex in examples]
    golds = [ex['metadata_gold_label'] for ex in examples]
    correct = np.array([int(is_correct(p, g)) for p, g in zip(preds, golds)])
    acc = float(correct.mean())
    ci_lo, ci_hi = bootstrap_ci(correct)
    # empty string rate
    empty_rate = float(sum(1 for p in preds if p == '') / n_total)
    per_condition[cond] = {
        'accuracy': acc,
        'ci_95_lo': ci_lo,
        'ci_95_hi': ci_hi,
        'n': n_total,
        'n_correct': int(correct.sum()),
        'empty_string_rate': empty_rate,
    }
    logger.info(f'  {SHORT_NAMES[cond]}: acc={acc:.4f} CI=[{ci_lo:.4f},{ci_hi:.4f}] empty={empty_rate:.3f}')

# Per-label breakdown for main method
main_pred = [ex.get('predict_main_method', '') for ex in examples]
labels = ['Entailment', 'Contradiction', 'Uncertain']
per_label_main = {}
for lbl in labels:
    idx = [i for i, g in enumerate(golds_all) if g == lbl]
    if idx:
        correct_lbl = [int(is_correct(main_pred[i], golds_all[i])) for i in idx]
        per_label_main[lbl] = {
            'accuracy': float(np.mean(correct_lbl)),
            'n': len(idx),
            'n_correct': int(sum(correct_lbl)),
        }
per_condition['predict_main_method']['per_label'] = per_label_main

09:54:34|INFO   |Computing per-condition accuracy + bootstrap CIs


09:54:34|INFO   |  Hetero-Oracle: acc=0.2222 CI=[0.1515,0.3030] empty=0.576


09:54:34|INFO   |  Top-1: acc=0.2727 CI=[0.1818,0.3636] empty=0.596


09:54:34|INFO   |  Self-Consist: acc=0.2626 CI=[0.1818,0.3535] empty=0.596


09:54:34|INFO   |  Direct-Judge: acc=0.2424 CI=[0.1616,0.3333] empty=0.576


09:54:34|INFO   |  Same-Oracle: acc=0.3131 CI=[0.2270,0.4093] empty=0.495


09:54:34|INFO   |  Rand-Worlds: acc=0.2828 CI=[0.2020,0.3737] empty=0.576


09:54:34|INFO   |  m=4 Worlds: acc=0.3030 CI=[0.2121,0.4040] empty=0.535


## Metric 2: McNemar's Test (15 Pairwise Comparisons, BH-corrected)

For each of the 15 condition pairs, runs McNemar's test on the discordant predictions. Uses the chi-squared approximation when `b+c >= 25`, otherwise exact binomial. Applies Benjamini-Hochberg FDR correction to all 15 p-values.

In [9]:
logger.info('Computing McNemar pairwise tests')
mcnemar_results = {}
raw_pvalues = []
pair_keys = []

golds_arr = np.array(golds_all)

for cA, cB in combinations(CONDITIONS, 2):
    pA_arr = np.array([ex.get(cA, '') for ex in examples])
    pB_arr = np.array([ex.get(cB, '') for ex in examples])

    # Exclude rows where either prediction is empty
    mask = (pA_arr != '') & (pB_arr != '')
    pA_m = pA_arr[mask]
    pB_m = pB_arr[mask]
    gA_m = golds_arr[mask]

    corrA = pA_m == gA_m
    corrB = pB_m == gA_m
    n00 = int(((~corrA) & (~corrB)).sum())
    n01 = int(((~corrA) & corrB).sum())   # B only correct
    n10 = int((corrA & (~corrB)).sum())    # A only correct
    n11 = int((corrA & corrB).sum())

    b, c = n10, n01  # discordant cells
    n_m = int(mask.sum())
    accA = float(corrA.mean()) if n_m > 0 else 0.0
    accB = float(corrB.mean()) if n_m > 0 else 0.0

    if b + c == 0:
        stat, pval = 0.0, 1.0
    elif b + c >= 25:
        # chi-squared approximation
        stat = (abs(b - c) - 1) ** 2 / (b + c)
        pval = float(1 - chi2.cdf(stat, df=1))
    else:
        # exact binomial (sign test)
        p_exact = binom.pmf(min(b, c), b + c, 0.5)
        # two-tailed
        pval = min(1.0, 2 * float(p_exact))
        stat = float((b - c) ** 2 / max(b + c, 1))

    h = cohen_h(accA, accB)
    key = f'{cA}__vs__{cB}'
    mcnemar_results[key] = {
        'condition_A': cA,
        'condition_B': cB,
        'n_valid': n_m,
        'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11,
        'chi2_stat': float(stat),
        'p_value_raw': float(pval),
        'cohen_h': float(h),
        'acc_A': accA,
        'acc_B': accB,
    }
    raw_pvalues.append(pval)
    pair_keys.append(key)

# BH correction
adjusted = bh_correction(raw_pvalues)
for key, adj_p in zip(pair_keys, adjusted):
    mcnemar_results[key]['p_value_adjusted'] = adj_p

# Log primary comparisons
primary = [
    ('predict_main_method', 'predict_top1_baseline', 'hetero-oracle vs top-1'),
    ('predict_main_method', 'predict_self_consistency', 'hetero-oracle vs self-consistency'),
    ('predict_main_method', 'predict_ablation_same_oracle', 'hetero-oracle vs same-model-oracle'),
]
for cA, cB, label in primary:
    key = f'{cA}__vs__{cB}'
    r = mcnemar_results[key]
    logger.info(f'  [{label}] chi2={r["chi2_stat"]:.3f} p_raw={r["p_value_raw"]:.4f} '
                f'p_adj={r["p_value_adjusted"]:.4f} h={r["cohen_h"]:.3f}')

09:54:34|INFO   |Computing McNemar pairwise tests


09:54:34|INFO   |  [hetero-oracle vs top-1] chi2=2.000 p_raw=0.5000 p_adj=0.9844 h=-0.136


09:54:34|INFO   |  [hetero-oracle vs self-consistency] chi2=4.000 p_raw=0.1250 p_adj=0.9844 h=-0.255


09:54:34|INFO   |  [hetero-oracle vs same-model-oracle] chi2=0.200 p_raw=0.6250 p_adj=0.9844 h=-0.056


## Metric 3: Normalized Self-Consistency

Alpha-renaming bound variables before Jaccard clustering tests whether variable naming artifacts affect self-consistency candidate selection. If the selected index changes after normalization, we conservatively report an empty prediction (since Z3 results are not available per-candidate).

In [10]:
logger.info('Computing normalized self-consistency')
norm_sc_preds = []
orig_sc_preds = [ex.get('predict_self_consistency', '') for ex in examples]
n_changed = 0
n_parse_fail = 0

for i, ex in enumerate(examples):
    cands_str = ex.get('metadata_candidates', '[]')
    try:
        candidates = json.loads(cands_str)
    except Exception:
        norm_sc_preds.append('')
        n_parse_fail += 1
        continue

    best_idx = cluster_and_pick(cands_str)
    if best_idx < 0 or best_idx >= len(candidates):
        norm_sc_preds.append('')
        n_parse_fail += 1
        continue

    # Simulate downstream prediction: we need to know what the original SC selected
    # We replicate the original cluster_and_pick on raw strings to find orig_idx
    # Then compare with best_idx from normalized version
    # For the normalized version, just use best_idx's original (un-executed) prediction
    # Since we don't have Z3 results per-candidate, we approximate:
    # If the best candidate index matches what orig SC would select, use orig SC prediction
    # Otherwise we can't know the downstream result without re-running Z3.
    # Strategy: if normalization changed the selected index, mark as "unknown" (empty)
    # but report the fraction changed.
    # Actually — the original SC index can be inferred from metadata_candidates and metadata_world_scores.
    # The original SC used raw Jaccard on premise+conclusion strings concatenated.
    # We re-cluster raw to find original index.

    norms = [normalize_fol_candidate(c) for c in candidates]
    # Original (raw) clustering — same as original code but without alpha-renaming
    raw_norms = []
    for c in candidates:
        parts = []
        for p in c.get('premises_fol', []):
            parts.append(str(p))
        parts.append(str(c.get('conclusion_fol', '')))
        raw_norms.append(re.sub(r'\s+', '', ' '.join(parts).lower()))

    # cluster on raw
    threshold = 0.3
    clusters_raw = []
    for idx2 in range(len(candidates)):
        placed = False
        for cl in clusters_raw:
            if any(jaccard_similarity(raw_norms[idx2], raw_norms[j]) >= threshold for j in cl):
                cl.append(idx2)
                placed = True
                break
        if not placed:
            clusters_raw.append([idx2])
    largest_ci_raw = max(range(len(clusters_raw)), key=lambda ci: (len(clusters_raw[ci]), -ci))
    orig_idx = min(clusters_raw[largest_ci_raw])  # centroid = lowest index in largest cluster

    if best_idx != orig_idx:
        n_changed += 1
        # Can't evaluate without Z3; use empty to be conservative
        norm_sc_preds.append('')
    else:
        norm_sc_preds.append(orig_sc_preds[i])

norm_sc_correct = np.array([int(is_correct(p, g)) for p, g in zip(norm_sc_preds, golds_all)])
norm_sc_acc = float(norm_sc_correct.mean())
orig_sc_correct = np.array([int(is_correct(p, g)) for p, g in zip(orig_sc_preds, golds_all)])
orig_sc_acc = float(orig_sc_correct.mean())
frac_changed = float(n_changed / n_total)

normalized_sc = {
    'normalized_sc_accuracy': norm_sc_acc,
    'original_sc_accuracy': orig_sc_acc,
    'delta_vs_original_sc': float(norm_sc_acc - orig_sc_acc),
    'fraction_changed': frac_changed,
    'n_changed': n_changed,
    'n_parse_fail': n_parse_fail,
}
logger.info(f'Normalized SC: acc={norm_sc_acc:.4f} orig={orig_sc_acc:.4f} '
            f'delta={norm_sc_acc-orig_sc_acc:.4f} changed={frac_changed:.3f}')

09:54:34|INFO   |Computing normalized self-consistency


09:54:34|INFO   |Normalized SC: acc=0.0000 orig=0.2626 delta=-0.2626 changed=0.000


## Metric 4: Uncertain-Class Decomposition

The FOLIO dataset has an 'Uncertain' class (examples with no definitive logical conclusion). This metric checks how many of the main method's Uncertain-class correct predictions are genuine vs. coincidence (empty parse that happens to match the 'Uncertain' gold label).

In [11]:
logger.info('Computing Uncertain-class decomposition')
gold_uncertain_idx = [i for i, g in enumerate(golds_all) if g == 'Uncertain']
n_gold_unc = len(gold_uncertain_idx)

main_preds = [ex.get('predict_main_method', '') for ex in examples]
unc_predicted_as_uncertain = sum(1 for i in gold_uncertain_idx if main_preds[i] == 'Uncertain')
unc_predicted_as_empty = sum(1 for i in gold_uncertain_idx if main_preds[i] == '')
unc_predicted_as_other = sum(1 for i in gold_uncertain_idx
                              if main_preds[i] not in ('Uncertain', '') and main_preds[i] != 'Uncertain')

# Coincidence hits: gold=Uncertain AND pred=Uncertain
coincidence_hits = unc_predicted_as_uncertain
total_main_correct_unc = per_condition['predict_main_method']['per_label'].get('Uncertain', {}).get('n_correct', 0)

uncertain_decomp = {
    'n_gold_uncertain': n_gold_unc,
    'main_predicted_uncertain_given_gold_uncertain': unc_predicted_as_uncertain,
    'main_predicted_empty_given_gold_uncertain': unc_predicted_as_empty,
    'main_predicted_wrong_given_gold_uncertain': unc_predicted_as_other,
    'coincidence_hits': coincidence_hits,
    'total_main_correct_uncertain': total_main_correct_unc,
    'coincidence_fraction_of_uncertain_accuracy': float(coincidence_hits / max(total_main_correct_unc, 1)),
}

# Empty string rates per condition across all examples
empty_rates = {c: per_condition[c]['empty_string_rate'] for c in CONDITIONS}
uncertain_decomp['empty_string_rates_all_conditions'] = empty_rates

logger.info(f'Gold-Uncertain ({n_gold_unc}): pred=Uncertain={unc_predicted_as_uncertain}, '
            f'pred=empty={unc_predicted_as_empty}, pred=wrong={unc_predicted_as_other}')

09:54:34|INFO   |Computing Uncertain-class decomposition


09:54:34|INFO   |Gold-Uncertain (33): pred=Uncertain=12, pred=empty=20, pred=wrong=1


## Metric 5: Cross-World Agreement AUC

Tests whether the maximum world agreement score (from the HOWP mechanism) is predictive of correctness. AUC-ROC measures discriminability; Kendall τ measures rank correlation. A significant AUC > 0.5 confirms the mechanism provides signal, even if it doesn't improve final accuracy above the beam recall ceiling.

In [12]:
logger.info('Computing cross-world agreement AUC')
world_scores_list = []
binary_correct = []
for ex in examples:
    try:
        ws = json.loads(ex['metadata_world_scores'])
        max_ws = float(max(ws))
    except Exception:
        max_ws = 0.5
    pred = ex.get('predict_main_method', '')
    gold = ex['metadata_gold_label']
    world_scores_list.append(max_ws)
    binary_correct.append(int(is_correct(pred, gold)))

ws_arr = np.array(world_scores_list)
bc_arr = np.array(binary_correct)

# AUC requires both classes present
if len(set(binary_correct)) < 2:
    auc_roc = float('nan')
    logger.warning('Only one class in binary_correct — AUC undefined')
else:
    auc_roc = float(roc_auc_score(bc_arr, ws_arr))

tau, tau_pval = kendalltau(ws_arr, bc_arr)

# Summary stats per group
correct_ws = ws_arr[bc_arr == 1]
wrong_ws = ws_arr[bc_arr == 0]

cross_world_auc = {
    'auc_roc': auc_roc,
    'kendall_tau': float(tau),
    'kendall_tau_pval': float(tau_pval),
    'n_correct': int(bc_arr.sum()),
    'n_incorrect': int((bc_arr == 0).sum()),
    'mean_max_ws_correct': float(correct_ws.mean()) if len(correct_ws) > 0 else float('nan'),
    'mean_max_ws_incorrect': float(wrong_ws.mean()) if len(wrong_ws) > 0 else float('nan'),
    'std_max_ws_correct': float(correct_ws.std()) if len(correct_ws) > 0 else float('nan'),
    'std_max_ws_incorrect': float(wrong_ws.std()) if len(wrong_ws) > 0 else float('nan'),
}
logger.info(f'Agreement AUC={auc_roc:.4f} tau={tau:.4f} (p={tau_pval:.4f})')

09:54:34|INFO   |Computing cross-world agreement AUC


09:54:34|INFO   |Agreement AUC=0.6638 tau=0.2231 (p=0.0152)


## Metric 6: Inter-Model Agreement Rate

Compares hetero-oracle (diverse oracle worlds) vs. homo-oracle (same-model worlds) to see how often they select the same candidate. If they mostly agree, the diversity in world generation is not actually changing candidate selection.

In [13]:
logger.info('Computing inter-model agreement rate')
hetero_preds = [ex.get('predict_main_method', '') for ex in examples]
homo_preds = [ex.get('predict_ablation_same_oracle', '') for ex in examples]

# Agreement on prediction (same label string, including empty)
agree = sum(1 for h, s in zip(hetero_preds, homo_preds) if h == s)
agree_rate = float(agree / n_total)

hetero_correct = np.array([int(is_correct(p, g)) for p, g in zip(hetero_preds, golds_all)])
homo_correct = np.array([int(is_correct(p, g)) for p, g in zip(homo_preds, golds_all)])

both_correct = int((hetero_correct & homo_correct).sum())
both_wrong = int(((1 - hetero_correct) & (1 - homo_correct)).sum())
only_hetero = int((hetero_correct & (1 - homo_correct)).sum())
only_homo = int(((1 - hetero_correct) & homo_correct).sum())

# Conditional accuracy: hetero correct given homo wrong
homo_wrong_idx = np.where(homo_correct == 0)[0]
cond_hetero_given_homo_wrong = float(hetero_correct[homo_wrong_idx].mean()) if len(homo_wrong_idx) > 0 else float('nan')

inter_model = {
    'inter_model_agreement_rate': agree_rate,
    'n_agree': agree,
    'both_correct': both_correct,
    'both_wrong': both_wrong,
    'only_hetero_correct': only_hetero,
    'only_homo_correct': only_homo,
    'conditional_accuracy_hetero_given_homo_wrong': cond_hetero_given_homo_wrong,
    'mcnemar_key': 'predict_main_method__vs__predict_ablation_same_oracle',
}
logger.info(f'Inter-model agreement={agree_rate:.4f} cond_acc_hetero|homo_wrong={cond_hetero_given_homo_wrong:.4f}')

09:54:34|INFO   |Computing inter-model agreement rate


09:54:34|INFO   |Inter-model agreement=0.7374 cond_acc_hetero|homo_wrong=0.0588


## Figures

Three figures visualizing the key results:
1. **Forest plot**: Accuracy + 95% CI for all 7 conditions
2. **Violin plot**: Max world agreement score for correct vs. incorrect predictions  
3. **McNemar heatmap**: -log10(BH-adjusted p-value) with Cohen's h annotations

In [14]:
os.makedirs('figures', exist_ok=True)

# --- Figure 1: CI Forest Plot ---
conditions = list(per_condition.keys())
# remove per_label key accidentally included
conditions = [c for c in conditions if c in CONDITIONS]
accuracies = [per_condition[c]['accuracy'] for c in conditions]
lows = [per_condition[c]['ci_95_lo'] for c in conditions]
highs = [per_condition[c]['ci_95_hi'] for c in conditions]
names = [SHORT_NAMES.get(c, c) for c in conditions]

order = np.argsort(accuracies)
conditions_s = [conditions[i] for i in order]
accuracies_s = [accuracies[i] for i in order]
lows_s = [lows[i] for i in order]
highs_s = [highs[i] for i in order]
names_s = [names[i] for i in order]

top1_acc = per_condition['predict_top1_baseline']['accuracy']

fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(conditions_s))
xerr_lo = [accuracies_s[i] - lows_s[i] for i in range(len(conditions_s))]
xerr_hi = [highs_s[i] - accuracies_s[i] for i in range(len(conditions_s))]

colors = ['#2196F3' if c == 'predict_main_method' else '#90CAF9' for c in conditions_s]
ax.barh(y, accuracies_s, xerr=[xerr_lo, xerr_hi], align='center',
        color=colors, ecolor='black', capsize=4, height=0.6)
ax.axvline(top1_acc, color='red', linestyle='--', linewidth=1.5, label=f'Top-1 baseline ({top1_acc:.3f})')
ax.set_yticks(y)
ax.set_yticklabels(names_s, fontsize=10)
ax.set_xlabel('Accuracy ± 95% Bootstrap CI', fontsize=11)
ax.set_title(f'Condition Accuracy Forest Plot (FOLIO, n={n_total})', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(0, 0.5)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/ci_forest_plot.png', dpi=150)
plt.show()
logger.info('Saved figures/ci_forest_plot.png')

# --- Figure 2: World Agreement Score Violin ---
correct_scores = []
wrong_scores = []
for ex in examples:
    try:
        ws = json.loads(ex['metadata_world_scores'])
    except Exception:
        continue
    max_ws = max(ws)
    pred = ex['predict_main_method']
    gold = ex['metadata_gold_label']
    if pred != '' and pred == gold:
        correct_scores.append(max_ws)
    else:
        wrong_scores.append(max_ws)

fig, ax = plt.subplots(figsize=(6, 5))
data_violin = [correct_scores, wrong_scores]
labels_violin = [f'Correct\n(n={len(correct_scores)})', f'Incorrect\n(n={len(wrong_scores)})']
parts = ax.violinplot(data_violin, positions=[1, 2], showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], ['#4CAF50', '#F44336']):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax.set_xticks([1, 2])
ax.set_xticklabels(labels_violin, fontsize=11)
ax.set_ylabel('Max World Agreement Score', fontsize=11)
ax.set_title('World Agreement Score: Correct vs Incorrect Selections', fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/agreement_score_violin.png', dpi=150)
plt.show()
logger.info('Saved figures/agreement_score_violin.png')

# --- Figure 3: McNemar Heatmap ---
n_cond = len(CONDITIONS)
pmat = np.full((n_cond, n_cond), np.nan)
hmat = np.full((n_cond, n_cond), np.nan)

for key, vals in mcnemar_results.items():
    a, b = key.split('__vs__')
    if a in CONDITIONS and b in CONDITIONS:
        i, j = CONDITIONS.index(a), CONDITIONS.index(b)
        p_adj = vals.get('p_value_adjusted', 1.0)
        h = vals.get('cohen_h', 0.0)
        pmat[i, j] = -math.log10(max(p_adj, 1e-10))
        hmat[i, j] = h
        pmat[j, i] = pmat[i, j]
        hmat[j, i] = -h

short = [SHORT_NAMES.get(c, c) for c in CONDITIONS]
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pmat, cmap='YlOrRd', vmin=0, aspect='auto')
plt.colorbar(im, ax=ax, label='-log10(adjusted p-value)')
ax.set_xticks(range(n_cond))
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(n_cond))
ax.set_yticklabels(short, fontsize=9)
ax.set_title('McNemar Pairwise Test: -log10(BH-adjusted p-value)\nAnnotated with Cohen\'s h', fontsize=11)

for i in range(n_cond):
    for j in range(n_cond):
        if not np.isnan(hmat[i, j]):
            ax.text(j, i, f'{hmat[i, j]:.2f}', ha='center', va='center',
                    fontsize=7, color='black')

plt.tight_layout()
plt.savefig('figures/mcnemar_pvalue_heatmap.png', dpi=150)
plt.show()
logger.info('Saved figures/mcnemar_pvalue_heatmap.png')

09:54:34|INFO   |Saved figures/ci_forest_plot.png


09:54:34|INFO   |Saved figures/agreement_score_violin.png


09:54:35|INFO   |Saved figures/mcnemar_pvalue_heatmap.png


## Results Summary

All key metrics in a readable table.

In [15]:
print('=' * 65)
print(f'HOWP FOLIO Statistical Re-Analysis  (n={n_total} examples)')
print('=' * 65)

print('\n--- Metric 1: Per-Condition Accuracy + 95% Bootstrap CIs ---')
print(f'{"Condition":<20} {"Acc":>6} {"CI-lo":>6} {"CI-hi":>6} {"Empty%":>7}')
print('-' * 50)
for cond in CONDITIONS:
    r = per_condition[cond]
    print(f'{SHORT_NAMES[cond]:<20} {r["accuracy"]:>6.3f} {r["ci_95_lo"]:>6.3f} {r["ci_95_hi"]:>6.3f} {r["empty_string_rate"]:>7.1%}')

print('\n--- Metric 2: Primary McNemar Comparisons (BH-corrected) ---')
print(f'{"Pair":<40} {"chi2":>6} {"p_raw":>7} {"p_adj":>7} {"h":>6}')
print('-' * 70)
for cA, cB, label in primary:
    key = f'{cA}__vs__{cB}'
    r = mcnemar_results[key]
    print(f'{label:<40} {r["chi2_stat"]:>6.3f} {r["p_value_raw"]:>7.4f} {r["p_value_adjusted"]:>7.4f} {r["cohen_h"]:>6.3f}')

print('\n--- Metric 3: Normalized Self-Consistency ---')
print(f'  Normalized SC accuracy:  {normalized_sc["normalized_sc_accuracy"]:.4f}')
print(f'  Original SC accuracy:    {normalized_sc["original_sc_accuracy"]:.4f}')
print(f'  Delta:                   {normalized_sc["delta_vs_original_sc"]:+.4f}')
print(f'  Fraction changed:        {normalized_sc["fraction_changed"]:.1%}')

print('\n--- Metric 4: Uncertain-Class Decomposition ---')
n_gu = uncertain_decomp['n_gold_uncertain']
print(f'  Gold-Uncertain examples:      {n_gu}')
print(f'  Pred=Uncertain (coincidence): {uncertain_decomp["main_predicted_uncertain_given_gold_uncertain"]}')
print(f'  Pred=Empty (parse fail):      {uncertain_decomp["main_predicted_empty_given_gold_uncertain"]}')
print(f'  Pred=Wrong:                   {uncertain_decomp["main_predicted_wrong_given_gold_uncertain"]}')
print(f'  Coincidence fraction:         {uncertain_decomp["coincidence_fraction_of_uncertain_accuracy"]:.1%}')

print('\n--- Metric 5: Cross-World Agreement AUC ---')
print(f'  AUC-ROC:          {cross_world_auc["auc_roc"]:.4f}')
print(f'  Kendall tau:      {cross_world_auc["kendall_tau"]:.4f}  (p={cross_world_auc["kendall_tau_pval"]:.4f})')
print(f'  Mean score (correct):   {cross_world_auc["mean_max_ws_correct"]:.3f}')
print(f'  Mean score (incorrect): {cross_world_auc["mean_max_ws_incorrect"]:.3f}')

print('\n--- Metric 6: Inter-Model Agreement ---')
print(f'  Hetero/homo agree rate:     {inter_model["inter_model_agreement_rate"]:.1%} ({inter_model["n_agree"]}/{n_total})')
print(f'  Both correct:               {inter_model["both_correct"]}')
print(f'  Both wrong:                 {inter_model["both_wrong"]}')
print(f'  Only hetero correct:        {inter_model["only_hetero_correct"]}')
print(f'  Only homo correct:          {inter_model["only_homo_correct"]}')
print(f'  Cond. acc (hetero|homo⊗):  {inter_model["conditional_accuracy_hetero_given_homo_wrong"]:.1%}')
print('=' * 65)

HOWP FOLIO Statistical Re-Analysis  (n=99 examples)

--- Metric 1: Per-Condition Accuracy + 95% Bootstrap CIs ---
Condition               Acc  CI-lo  CI-hi  Empty%
--------------------------------------------------
Hetero-Oracle         0.222  0.152  0.303   57.6%
Top-1                 0.273  0.182  0.364   59.6%
Self-Consist          0.263  0.182  0.354   59.6%
Direct-Judge          0.242  0.162  0.333   57.6%
Same-Oracle           0.313  0.227  0.409   49.5%
Rand-Worlds           0.283  0.202  0.374   57.6%
m=4 Worlds            0.303  0.212  0.404   53.5%

--- Metric 2: Primary McNemar Comparisons (BH-corrected) ---
Pair                                       chi2   p_raw   p_adj      h
----------------------------------------------------------------------
hetero-oracle vs top-1                    2.000  0.5000  0.9844 -0.136
hetero-oracle vs self-consistency         4.000  0.1250  0.9844 -0.255
hetero-oracle vs same-model-oracle        0.200  0.6250  0.9844 -0.056

--- Metric 3: Nor